In [1]:
import { load } from "dotenv";
const env = await load();

const process = {
    env
}

In [2]:
env

{ DASHSCOPE_API_KEY: "sk-2be7ec77a2cd4c2a9e9a6ccfe2fe3c97" }

In [3]:
import { TextLoader } from "langchain/document_loaders/fs/text";
import { RecursiveCharacterTextSplitter } from "langchain/text_splitter";
const loader = new TextLoader("data/kong.txt");
const docs = await loader.load();

const splitter = new RecursiveCharacterTextSplitter({
    chunkSize: 100,
    chunkOverlap: 20,
  });

const splitDocs = await splitter.splitDocuments(docs);

In [ ]:
splitDocs

In [ ]:
console.log(splitDocs[0])

In [6]:
import { AlibabaTongyiEmbeddings } from "@langchain/community/embeddings/alibaba_tongyi";

const model = new AlibabaTongyiEmbeddings({ 
    apiKey: process.env.DASHSCOPE_API_KEY,
    modelName: "text-embedding-v2",
});
const res = await model.embedQuery(splitDocs[0].pageContent);

In [7]:
res

[
    0.005916495048378137,   0.006229275810535023,    0.04032567131471087,
   -0.018727336580509096,   -0.03921941514539809,   0.011457653182168013,
    0.014723742824900963,    0.04577793386346668, -0.0026816834818608752,
     0.01072014906929283,  -0.002650405405645187,   0.017844965588319144,
   -0.004780607017387343,  -0.008573485312173995,   -0.04180067954046123,
    0.011714462650044192,  -0.008428618432859228,  -0.032134107775275804,
    0.035189481957187276,  -0.013156546585041201,    0.02122958267776418,
    0.033951528624861074,   0.025904831964740783,   0.005346904818345073,
     0.05220475541852184,   0.010094587544979059,  -0.006028437636939549,
    -0.01796349303503123,   0.033372061107602005,   0.030685438982128124,
    0.011918593252715002,    0.01455253651298351,   0.004727928152181973,
     0.02362647104460852,  -0.004329544234066361,   0.007328947121697125,
   -0.007328947121697125,   0.000570413337301899,  -0.009818023502650866,
  -0.0030504355382984663,   0.011977

In [9]:
import { MemoryVectorStore } from "langchain/vectorstores/memory";

const vectorstore = new MemoryVectorStore(model);
await vectorstore.addDocuments(splitDocs);


In [ ]:
// 直接从 vector store 的实例中自动生成，这里我们传入了参数 2，代表对应每个输入，我们想要返回相似度最高的两个文本内容
const retriever = vectorstore.asRetriever(2)

In [11]:
const res = await retriever.invoke("茴香豆是做什么用的")

In [12]:
res

[
  Document {
    pageContent: "有几回，邻居孩子听得笑声，也赶热闹，围住了孔乙己。他便给他们一人一颗。孩子吃完豆，仍然不散，眼睛都望着碟子。孔乙己着了慌，伸开五指将碟子罩住，弯腰下去说道，“不多了，我已经不多了。”直起身又看一看豆",
    metadata: { source: "data/kong.txt", loc: { lines: { from: 15, to: 15 } } }
  },
  Document {
    pageContent: "年前的事，现在每碗要涨到十文，——靠柜外站着，热热的喝了休息；倘肯多花一文，便可以买一碟盐煮笋，或者茴香豆，做下酒物了，如果出到十几文，那就能买一样荤菜，但这些顾客，多是短衣帮，大抵没有这样阔绰。只有",
    metadata: { source: "data/kong.txt", loc: { lines: { from: 1, to: 1 } } }
  }
]

In [13]:
const res = await retriever.invoke("下酒菜一般是什么？")
res

[
  Document {
    pageContent: "年前的事，现在每碗要涨到十文，——靠柜外站着，热热的喝了休息；倘肯多花一文，便可以买一碟盐煮笋，或者茴香豆，做下酒物了，如果出到十几文，那就能买一样荤菜，但这些顾客，多是短衣帮，大抵没有这样阔绰。只有",
    metadata: { source: "data/kong.txt", loc: { lines: { from: 1, to: 1 } } }
  },
  Document {
    pageContent: "。他们往往要亲眼看着黄酒从坛子里舀出，看过壶子底里有水没有，又亲看将壶子放在热水里，然后放心：在这严重监督下，羼水也很为难。所以过了几天，掌柜又说我干不了这事。幸亏荐头的情面大，辞退不得，便改为专管温",
    metadata: { source: "data/kong.txt", loc: { lines: { from: 3, to: 3 } } }
  }
]

In [ ]:
const res = await retriever.invoke("孔乙己用什么谋生？")
res

[
  Document {
    pageContent: "孔乙己喝过半碗酒，涨红的脸色渐渐复了原，旁人便又问道，“孔乙己，你当真认识字么？”孔乙己看着问他的人，显出不屑置辩的神气。他们便接着说道，“你怎的连半个秀才也捞不到呢？”孔乙己立刻显出颓唐不安模样，",
    metadata: { source: "data/kong.txt", loc: { lines: { from: 11, to: 11 } } }
  },
  Document {
    pageContent: "孔乙己是这样的使人快活，可是没有他，别人也便这么过。",
    metadata: { source: "data/kong.txt", loc: { lines: { from: 17, to: 17 } } }
  }
]

: 